# Document Classification Pipeline - Full End-to-End Verification

This notebook demonstrates the complete lifecycle of a document within the Enterprise Knowledge Base (EKB):
1. **Step 0: Ingestion**: Uploading a local PDF with custom metadata to the Landing Zone.
2. **Step 1: DLP Security Gate**: Deterministic scanning and automated masking (for high-risk content).
3. **Step 2: Contextual Classification**: Multimodal Gemini 2.5 Flash reasoning over the (masked) document.
4. **Step 3: Routing & Persistence**: Moving original/masked files to domain buckets and indexing metadata in BigQuery.

In [ ]:
import sys
import os
from loguru import logger
from google.cloud import storage, bigquery

# Ensure repo root is in path
repo_root = os.path.abspath(os.path.join(os.getcwd(), "../../.."))
if repo_root not in sys.path:
    sys.path.append(repo_root)

from pipelines.enterprise_knowledge_base import ClassificationPipeline
from pipelines.enterprise_knowledge_base.document_classification.config import EKB_CONFIG
from pipelines.enterprise_knowledge_base.document_classification.gcs_service import GCSService
from pipelines.enterprise_knowledge_base.document_classification.schemas import FileRoutingRequest, MetadataBQRequest

print(f"Project ID: {EKB_CONFIG.PROJECT_ID}")
print("Services imported.")

## 1. Step 0: Ingestion (Local File to Landing Zone)
We will upload a local PDF and attach the custom metadata expected by the pipeline.

In [ ]:
def ingest_local_file(local_path: str, metadata: dict) -> str:
    """Simulates the Agent ADK Skill: Uploads a local file with metadata to GCS.
    
    This helper function is self-contained to avoid modifying core service classes.
    
    Args:
        local_path: Path to the local file.
        metadata: Dictionary of metadata (project, domain, trust-level, etc.)
        
    Returns:
        str: The landing zone GCS URI.
    """
    client = storage.Client()
    bucket_name = "enterprise_knowledgebase_landing_zone"
    filename = os.path.basename(local_path)
    bucket = client.bucket(bucket_name)
    blob = bucket.blob(filename)
    
    # Attach custom metadata (x-goog-meta-*)
    blob.metadata = metadata
    
    print(f"Uploading {filename} to gs://{bucket_name}/{filename} with metadata: {metadata}")
    blob.upload_from_filename(local_path, content_type="application/pdf")
    
    return f"gs://{bucket_name}/{filename}"

# Metadata from Design.md
sample_metadata = {
    "project": "alpha-strategy-2026",
    "domain": "finance",
    "trust-level": "wip",
    "uploader": "emmanuel.amador@midominio.com",
    "creator-name": "Emmanuel Amador"
}

# TODO: Change this to your local PDF path
local_pdf_path = "test_audit.pdf" 

if os.path.exists(local_pdf_path):
    landing_zone_uri = ingest_local_file(local_pdf_path, sample_metadata)
    print(f"Ingestion successful: {landing_zone_uri}")
else:
    print(f"ERROR: Local file {local_pdf_path} not found. Please provide a valid path.")

## 2. Step 1: DLP Security Gate (Scan & Mask)
Deterministic phase to detect high-risk data and protect privacy.

In [ ]:
pipeline = ClassificationPipeline()

print(f"Triggering DLP Phase for: {landing_zone_uri}")
dlp_result = pipeline.dlp_trigger(landing_zone_uri)

print(f"DLP Result URI: {dlp_result.sanitized_gcs_uri}")
print(f"Proposed Tier: {dlp_result.proposed_classification_tier}")

## 3. Step 2: Contextual Classification (Gemini 2.5 Flash)
Probabilistic phase to refine the tier and domain using multimodal reasoning.

In [ ]:
print("Triggering Gemini Contextual Phase...")
# Use the metadata extracted from GCS for Phase 2
blob_meta = pipeline._get_blob_metadata(landing_zone_uri)

verdict = pipeline.contextual_classification(
    sanitized_url=dlp_result.sanitized_gcs_uri,
    proposed_classification_tier=dlp_result.proposed_classification_tier,
    proposed_domain=blob_meta.proposed_domain,
    trust_level=blob_meta.trust_level
)

print("\n--- Gemini Verdict ---")
print(f"Final Tier: {verdict.final_classification_tier}")
print(f"Confidence: {verdict.confidence}")
print(f"Validated Domain: {verdict.final_domain}")
print(f"Summary: {verdict.file_description}")

## 4. Step 3: Routing & BQ Persistence
Relocation to domain buckets and BigQuery indexing.

In [ ]:
print("Executing final routing and BQ persistence...")

routing_req = FileRoutingRequest(
    original_landing_uri=landing_zone_uri,
    sanitized_landing_uri=dlp_result.sanitized_gcs_uri,
    final_domain=verdict.final_domain,
    final_security_tier=verdict.final_classification_tier,
    project_name=blob_meta.project_name or "unknown",
    uploader_email=blob_meta.uploader_email or "unknown"
)

routing_res = pipeline.file_routing(routing_req)

bq_req = MetadataBQRequest(
    final_original_uri=routing_res.final_original_uri,
    final_sanitized_uri=routing_res.final_sanitized_uri,
    llm_classification=verdict,
    blob_metadata=blob_meta
)

pipeline.metadata_bq(bq_req)

print("\n--- Pipeline Complete ---")
print(f"Original File Secured at: {routing_res.final_original_uri}")
if routing_res.final_sanitized_uri:
    print(f"Masked File Secured at: {routing_res.final_sanitized_uri}")
print("Metadata indexed in BigQuery knowledge_base.documents_metadata.")